# Hassan Project 4 — Stock Market Analytics & Time Series Forecasting

**Python + SQL + Time Series + Power BI**

This notebook is designed to run **top-to-bottom in Google Colab**.

### Market universe
- AAPL
- MSFT
- NVDA
- AMZN
- GOOGL
- META
- SPY (benchmark)

### Fixed historical period
**1 January 2020 to 31 December 2025**

### What this notebook does
- Downloads historical OHLCV market data
- Cleans and validates the data
- Engineers returns, moving averages, volatility and drawdown features
- Creates financial KPI summaries
- Builds a SQLite database and runs SQL analysis
- Generates MySQL-ready SQL scripts
- Compares securities by return and risk
- Builds correlation analysis
- Forecasts SPY using a naive baseline and ARIMA
- Evaluates MAE, RMSE and MAPE
- Creates future 20-business-day model forecasts
- Exports Power BI-ready files
- Saves charts, reports and model output
- Creates one ZIP containing all project outputs

> Run using **Runtime → Run all**.

## 1. Install and import libraries

`yfinance` is installed automatically in Colab. `statsmodels` is used for ARIMA forecasting.

In [ ]:
!pip -q install yfinance

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sqlite3
import joblib
import yfinance as yf

from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_DIR = Path("project_4_outputs")
DATA_DIR = PROJECT_DIR / "data"
IMAGES_DIR = PROJECT_DIR / "images"
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"
POWERBI_DIR = PROJECT_DIR / "powerbi"
SQL_DIR = PROJECT_DIR / "sql"
SRC_DIR = PROJECT_DIR / "src"

for folder in [
    DATA_DIR, IMAGES_DIR, MODELS_DIR, REPORTS_DIR,
    POWERBI_DIR, SQL_DIR, SRC_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "SPY"]
START_DATE = "2020-01-01"
END_DATE = "2026-01-01"  # yfinance end date is exclusive
PRIMARY_TICKER = "SPY"

print("Folders created.")
print("Tickers:", TICKERS)
print("Historical period: 2020-01-01 to 2025-12-31")

## 2. Download historical market data

We download each ticker separately to keep the output structure simple and reproducible.

In [ ]:
def download_ticker(ticker):
    df = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=False,
        progress=False,
        threads=False
    )

    if df.empty:
        raise RuntimeError(f"No data returned for {ticker}")

    # yfinance can sometimes return MultiIndex columns.
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df = df.reset_index()
    df["Ticker"] = ticker

    # Keep expected fields.
    expected = ["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume", "Ticker"]
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise RuntimeError(f"{ticker}: Missing columns {missing}")

    return df[expected]

frames = []
for ticker in TICKERS:
    print("Downloading:", ticker)
    frames.append(download_ticker(ticker))

raw_df = pd.concat(frames, ignore_index=True)

print("\nRaw shape:", raw_df.shape)
print("Date range:", raw_df["Date"].min(), "to", raw_df["Date"].max())
display(raw_df.head())

## 3. Data validation and cleaning

Checks include:
- duplicate ticker/date rows
- missing values
- non-positive prices
- negative volume
- date ordering

In [ ]:
df = raw_df.copy()

df["Date"] = pd.to_datetime(df["Date"])

print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate ticker/date rows:", df.duplicated(subset=["Ticker", "Date"]).sum())

print("\nMissing values before cleaning:")
display(df.isna().sum().to_frame("missing"))

# Remove exact / ticker-date duplicates.
df = df.drop_duplicates()
df = df.drop_duplicates(subset=["Ticker", "Date"], keep="last")

# Validate numeric ranges.
price_cols = ["Open", "High", "Low", "Close", "Adj Close"]
valid_prices = (df[price_cols] > 0).all(axis=1)
valid_volume = df["Volume"].fillna(0) >= 0

df = df[valid_prices & valid_volume].copy()

# Drop rows with missing essential values.
df = df.dropna(subset=["Date", "Ticker", "Adj Close", "Close", "Volume"]).copy()

df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print("Clean shape:", df.shape)
print("Remaining missing values:", int(df.isna().sum().sum()))
print("Ticker/date duplicates after cleaning:", df.duplicated(subset=["Ticker", "Date"]).sum())

coverage = df.groupby("Ticker")["Date"].agg(["min", "max", "count"])
display(coverage)

## 4. Feature engineering

Financial features:
- Daily return
- Log return
- Cumulative return
- 20-day / 50-day moving averages
- 20-day rolling volatility
- Running peak
- Drawdown
- Volume percentage change
- Calendar fields

In [ ]:
def engineer_group(g):
    g = g.sort_values("Date").copy()

    g["DailyReturn"] = g["Adj Close"].pct_change()
    g["LogReturn"] = np.log(g["Adj Close"] / g["Adj Close"].shift(1))

    g["CumulativeReturn"] = (1 + g["DailyReturn"].fillna(0)).cumprod() - 1

    g["MA20"] = g["Adj Close"].rolling(20).mean()
    g["MA50"] = g["Adj Close"].rolling(50).mean()

    g["RollingVol20"] = g["DailyReturn"].rolling(20).std() * np.sqrt(252)

    g["RunningPeak"] = g["Adj Close"].cummax()
    g["Drawdown"] = g["Adj Close"] / g["RunningPeak"] - 1

    g["VolumeChange"] = g["Volume"].pct_change()

    return g

df = (
    df.groupby("Ticker", group_keys=False)
      .apply(engineer_group)
      .reset_index(drop=True)
)

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.to_period("M").astype(str)
df["MonthName"] = df["Date"].dt.month_name()
df["DayOfWeek"] = df["Date"].dt.day_name()

display(df.head(25))

## 5. Financial KPI summary

We calculate:
- Start / end price
- Total return
- Annualised return
- Annualised volatility
- Maximum drawdown
- Average daily volume
- Best / worst daily return
- Positive-day percentage
- Sharpe-style ratio using a **0% risk-free-rate assumption**

In [ ]:
kpi_rows = []

for ticker, g in df.groupby("Ticker"):
    g = g.sort_values("Date").copy()
    returns = g["DailyReturn"].dropna()

    start_price = g.iloc[0]["Adj Close"]
    end_price = g.iloc[-1]["Adj Close"]
    total_return = end_price / start_price - 1

    years = max((g["Date"].iloc[-1] - g["Date"].iloc[0]).days / 365.25, 1/365.25)
    annualized_return = (1 + total_return) ** (1 / years) - 1
    annualized_volatility = returns.std() * np.sqrt(252)

    max_drawdown = g["Drawdown"].min()
    avg_volume = g["Volume"].mean()
    best_day = returns.max()
    worst_day = returns.min()
    positive_day_pct = (returns > 0).mean()

    sharpe_style = (
        annualized_return / annualized_volatility
        if annualized_volatility and not np.isnan(annualized_volatility)
        else np.nan
    )

    kpi_rows.append({
        "Ticker": ticker,
        "StartDate": g["Date"].min(),
        "EndDate": g["Date"].max(),
        "StartAdjClose": start_price,
        "EndAdjClose": end_price,
        "TotalReturn": total_return,
        "AnnualizedReturn": annualized_return,
        "AnnualizedVolatility": annualized_volatility,
        "MaximumDrawdown": max_drawdown,
        "AverageDailyVolume": avg_volume,
        "BestDailyReturn": best_day,
        "WorstDailyReturn": worst_day,
        "PositiveDayPct": positive_day_pct,
        "SharpeStyleRatio_0pctRF": sharpe_style
    })

kpi_df = pd.DataFrame(kpi_rows).sort_values("TotalReturn", ascending=False)

display(
    kpi_df.style.format({
        "StartAdjClose": "${:,.2f}",
        "EndAdjClose": "${:,.2f}",
        "TotalReturn": "{:.2%}",
        "AnnualizedReturn": "{:.2%}",
        "AnnualizedVolatility": "{:.2%}",
        "MaximumDrawdown": "{:.2%}",
        "AverageDailyVolume": "{:,.0f}",
        "BestDailyReturn": "{:.2%}",
        "WorstDailyReturn": "{:.2%}",
        "PositiveDayPct": "{:.2%}",
        "SharpeStyleRatio_0pctRF": "{:.2f}"
    })
)

kpi_df.to_csv(DATA_DIR / "market_kpi_summary.csv", index=False)

## 6. Core market visualisations

In [ ]:
# Cumulative returns
plt.figure(figsize=(12, 6))
for ticker, g in df.groupby("Ticker"):
    plt.plot(g["Date"], g["CumulativeReturn"] * 100, label=ticker)

plt.title("Cumulative Returns — 2020 to 2025")
plt.ylabel("Cumulative Return (%)")
plt.xlabel("Date")
plt.legend(ncol=4)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "cumulative_returns.png", dpi=160, bbox_inches="tight")
plt.show()

# Annualised risk-return scatter
plt.figure(figsize=(8, 6))
plt.scatter(
    kpi_df["AnnualizedVolatility"] * 100,
    kpi_df["AnnualizedReturn"] * 100
)
for _, row in kpi_df.iterrows():
    plt.annotate(
        row["Ticker"],
        (row["AnnualizedVolatility"] * 100, row["AnnualizedReturn"] * 100),
        xytext=(5, 5),
        textcoords="offset points"
    )

plt.title("Annualised Return vs Volatility")
plt.xlabel("Annualised Volatility (%)")
plt.ylabel("Annualised Return (%)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "volatility_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

## 7. Drawdown analysis

In [ ]:
plt.figure(figsize=(12, 6))
for ticker, g in df.groupby("Ticker"):
    plt.plot(g["Date"], g["Drawdown"] * 100, label=ticker)

plt.title("Historical Drawdown")
plt.ylabel("Drawdown (%)")
plt.xlabel("Date")
plt.legend(ncol=4)
plt.tight_layout()
plt.savefig(IMAGES_DIR / "drawdown.png", dpi=160, bbox_inches="tight")
plt.show()

## 8. Return correlation matrix

In [ ]:
return_wide = (
    df.pivot(index="Date", columns="Ticker", values="DailyReturn")
      .sort_index()
)

corr_df = return_wide.corr()
display(corr_df.round(3))

corr_df.to_csv(DATA_DIR / "correlation_matrix.csv")

plt.figure(figsize=(8, 7))
plt.imshow(corr_df.values, aspect="auto")
plt.xticks(range(len(corr_df.columns)), corr_df.columns)
plt.yticks(range(len(corr_df.index)), corr_df.index)
plt.colorbar(label="Correlation")

for i in range(len(corr_df.index)):
    for j in range(len(corr_df.columns)):
        plt.text(j, i, f"{corr_df.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.title("Daily Return Correlation Matrix")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "correlation_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

## 9. Yearly performance analysis

In [ ]:
yearly_rows = []

for (ticker, year), g in df.groupby(["Ticker", "Year"]):
    g = g.sort_values("Date")
    yearly_return = g.iloc[-1]["Adj Close"] / g.iloc[0]["Adj Close"] - 1

    yearly_rows.append({
        "Ticker": ticker,
        "Year": year,
        "YearlyReturn": yearly_return,
        "AverageVolume": g["Volume"].mean(),
        "AnnualizedVolatility": g["DailyReturn"].std() * np.sqrt(252)
    })

yearly_df = pd.DataFrame(yearly_rows)
display(
    yearly_df.pivot(index="Ticker", columns="Year", values="YearlyReturn")
             .style.format("{:.2%}")
)

yearly_df.to_csv(DATA_DIR / "yearly_performance.csv", index=False)

## 10. Save cleaned market dataset

In [ ]:
clean_cols = [
    "Date", "Ticker",
    "Open", "High", "Low", "Close", "Adj Close", "Volume",
    "DailyReturn", "LogReturn", "CumulativeReturn",
    "MA20", "MA50", "RollingVol20",
    "RunningPeak", "Drawdown", "VolumeChange",
    "Year", "Month", "MonthName", "DayOfWeek"
]

clean_df = df[clean_cols].copy()

clean_path = DATA_DIR / "market_data_cleaned.csv"
clean_df.to_csv(clean_path, index=False)

print("Saved:", clean_path)
print("Rows:", len(clean_df))

## 11. SQLite database and SQL analysis

SQLite is used in Colab because it requires no separate database server.

The notebook also generates **MySQL-ready SQL scripts** for GitHub.

In [ ]:
db_path = PROJECT_DIR / "market_analytics.sqlite"

conn = sqlite3.connect(db_path)

sql_df = clean_df.copy()
sql_df["Date"] = sql_df["Date"].dt.strftime("%Y-%m-%d")

sql_df.to_sql("market_data", conn, if_exists="replace", index=False)
kpi_df.to_sql("market_kpi_summary", conn, if_exists="replace", index=False)
yearly_df.to_sql("yearly_performance", conn, if_exists="replace", index=False)

print("SQLite database created:", db_path)

## 12. Run 20 SQL analytical questions

In [ ]:
SQL_QUERIES = {
    "01_date_coverage": '''
        SELECT
            Ticker,
            MIN(Date) AS FirstTradingDate,
            MAX(Date) AS LastTradingDate,
            COUNT(*) AS TradingDays
        FROM market_data
        GROUP BY Ticker
        ORDER BY Ticker;
    ''',

    "02_latest_price": '''
        WITH ranked AS (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY Ticker ORDER BY Date DESC
                   ) AS rn
            FROM market_data
        )
        SELECT Ticker, Date, "Adj Close" AS LatestAdjClose
        FROM ranked
        WHERE rn = 1
        ORDER BY Ticker;
    ''',

    "03_total_return_ranking": '''
        SELECT
            Ticker,
            TotalReturn
        FROM market_kpi_summary
        ORDER BY TotalReturn DESC;
    ''',

    "04_volatility_ranking": '''
        SELECT
            Ticker,
            AnnualizedVolatility
        FROM market_kpi_summary
        ORDER BY AnnualizedVolatility DESC;
    ''',

    "05_max_drawdown": '''
        SELECT
            Ticker,
            MaximumDrawdown
        FROM market_kpi_summary
        ORDER BY MaximumDrawdown ASC;
    ''',

    "06_best_worst_days": '''
        SELECT
            Ticker,
            MAX(DailyReturn) AS BestDailyReturn,
            MIN(DailyReturn) AS WorstDailyReturn
        FROM market_data
        GROUP BY Ticker
        ORDER BY Ticker;
    ''',

    "07_yearly_returns": '''
        SELECT
            Ticker,
            Year,
            YearlyReturn
        FROM yearly_performance
        ORDER BY Ticker, Year;
    ''',

    "08_monthly_average_returns": '''
        SELECT
            Ticker,
            Month,
            AVG(DailyReturn) AS AvgDailyReturn
        FROM market_data
        WHERE DailyReturn IS NOT NULL
        GROUP BY Ticker, Month
        ORDER BY Ticker, AvgDailyReturn DESC;
    ''',

    "09_average_volume": '''
        SELECT
            Ticker,
            AVG(Volume) AS AverageVolume
        FROM market_data
        GROUP BY Ticker
        ORDER BY AverageVolume DESC;
    ''',

    "10_month_end_20day_return_proxy": '''
        SELECT
            Ticker,
            Month,
            MAX(CumulativeReturn) - MIN(CumulativeReturn) AS MonthCumulativeRange
        FROM market_data
        GROUP BY Ticker, Month
        ORDER BY MonthCumulativeRange DESC;
    ''',

    "11_positive_negative_days": '''
        SELECT
            Ticker,
            SUM(CASE WHEN DailyReturn > 0 THEN 1 ELSE 0 END) AS PositiveDays,
            SUM(CASE WHEN DailyReturn < 0 THEN 1 ELSE 0 END) AS NegativeDays,
            SUM(CASE WHEN DailyReturn = 0 THEN 1 ELSE 0 END) AS FlatDays
        FROM market_data
        GROUP BY Ticker;
    ''',

    "12_days_above_ma50": '''
        SELECT
            Ticker,
            SUM(CASE WHEN "Adj Close" > MA50 THEN 1 ELSE 0 END) AS DaysAboveMA50,
            COUNT(MA50) AS DaysWithMA50,
            1.0 * SUM(CASE WHEN "Adj Close" > MA50 THEN 1 ELSE 0 END)
                / NULLIF(COUNT(MA50), 0) AS PctAboveMA50
        FROM market_data
        GROUP BY Ticker
        ORDER BY PctAboveMA50 DESC;
    ''',

    "13_top_volume_days": '''
        SELECT
            Date, Ticker, Volume, "Adj Close"
        FROM market_data
        ORDER BY Volume DESC
        LIMIT 10;
    ''',

    "14_relative_to_spy": '''
        SELECT
            k.Ticker,
            k.TotalReturn,
            spy.TotalReturn AS SPYTotalReturn,
            k.TotalReturn - spy.TotalReturn AS ExcessVsSPY
        FROM market_kpi_summary k
        CROSS JOIN (
            SELECT TotalReturn
            FROM market_kpi_summary
            WHERE Ticker = 'SPY'
        ) spy
        ORDER BY ExcessVsSPY DESC;
    ''',

    "15_period_volatility": '''
        SELECT
            Ticker,
            CASE
                WHEN Date < '2022-01-01' THEN '2020-2021'
                WHEN Date < '2024-01-01' THEN '2022-2023'
                ELSE '2024-2025'
            END AS Period,
            AVG(RollingVol20) AS AvgRollingVolatility
        FROM market_data
        WHERE RollingVol20 IS NOT NULL
        GROUP BY Ticker, Period
        ORDER BY Ticker, Period;
    ''',

    "16_day_of_week_returns": '''
        SELECT
            Ticker,
            DayOfWeek,
            AVG(DailyReturn) AS AverageDailyReturn
        FROM market_data
        WHERE DailyReturn IS NOT NULL
        GROUP BY Ticker, DayOfWeek
        ORDER BY Ticker, AverageDailyReturn DESC;
    ''',

    "17_risk_adjusted_ranking": '''
        SELECT
            Ticker,
            AnnualizedReturn,
            AnnualizedVolatility,
            SharpeStyleRatio_0pctRF
        FROM market_kpi_summary
        ORDER BY SharpeStyleRatio_0pctRF DESC;
    ''',

    "18_large_move_days": '''
        SELECT
            Ticker,
            COUNT(*) AS TotalReturnDays,
            SUM(CASE WHEN ABS(DailyReturn) > 0.03 THEN 1 ELSE 0 END) AS LargeMoveDays,
            1.0 * SUM(CASE WHEN ABS(DailyReturn) > 0.03 THEN 1 ELSE 0 END)
                / COUNT(*) AS LargeMovePct
        FROM market_data
        WHERE DailyReturn IS NOT NULL
        GROUP BY Ticker
        ORDER BY LargeMovePct DESC;
    ''',

    "19_window_rank_total_return": '''
        SELECT
            Ticker,
            TotalReturn,
            RANK() OVER (ORDER BY TotalReturn DESC) AS ReturnRank
        FROM market_kpi_summary
        ORDER BY ReturnRank;
    ''',

    "20_powerbi_summary": '''
        SELECT
            Ticker,
            StartDate,
            EndDate,
            StartAdjClose,
            EndAdjClose,
            TotalReturn,
            AnnualizedReturn,
            AnnualizedVolatility,
            MaximumDrawdown,
            AverageDailyVolume,
            PositiveDayPct,
            SharpeStyleRatio_0pctRF
        FROM market_kpi_summary
        ORDER BY TotalReturn DESC;
    '''
}

SQL_RESULTS_DIR = PROJECT_DIR / "sql_results"
SQL_RESULTS_DIR.mkdir(exist_ok=True)

for name, query in SQL_QUERIES.items():
    result = pd.read_sql_query(query, conn)
    result.to_csv(SQL_RESULTS_DIR / f"{name}.csv", index=False)

print("Executed", len(SQL_QUERIES), "SQL analyses.")
display(pd.read_sql_query(SQL_QUERIES["03_total_return_ranking"], conn))

## 13. Forecasting setup — SPY

We use a chronological train/test split.

### Models
1. **Naive baseline:** tomorrow's price = today's price
2. **ARIMA(5,1,0)**

The last **60 trading days** are reserved for testing.

This is an educational forecasting exercise, **not financial advice**.

In [ ]:
spy = (
    df[df["Ticker"] == PRIMARY_TICKER][["Date", "Adj Close"]]
    .dropna()
    .sort_values("Date")
    .reset_index(drop=True)
)

TEST_DAYS = 60

train = spy.iloc[:-TEST_DAYS].copy()
test = spy.iloc[-TEST_DAYS:].copy()

print("Training observations:", len(train))
print("Test observations:", len(test))
print("Train end:", train["Date"].max())
print("Test start:", test["Date"].min())
print("Test end:", test["Date"].max())

## 14. Naive baseline forecast

In [ ]:
# Previous known price as one-step-ahead prediction.
naive_pred = test["Adj Close"].shift(1)
naive_pred.iloc[0] = train["Adj Close"].iloc[-1]

naive_predictions = naive_pred.to_numpy()

print("Naive forecast created.")

## 15. ARIMA forecast

We fit ARIMA on training prices and forecast the complete held-out 60-day test period.

In [ ]:
arima_order = (5, 1, 0)

arima_model = ARIMA(
    train["Adj Close"].astype(float),
    order=arima_order
).fit()

arima_predictions = arima_model.forecast(steps=len(test)).to_numpy()

print(arima_model.summary())

## 16. Forecast evaluation

Metrics:
- MAE
- RMSE
- MAPE

MAPE is safe here because stock prices are strictly positive.

In [ ]:
actual = test["Adj Close"].to_numpy()

def forecast_metrics(actual, pred, model_name):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE_Pct": mape
    }

forecast_metrics_df = pd.DataFrame([
    forecast_metrics(actual, naive_predictions, "Naive Previous Close"),
    forecast_metrics(actual, arima_predictions, "ARIMA(5,1,0)")
]).sort_values("RMSE")

display(forecast_metrics_df.style.format({
    "MAE": "${:,.2f}",
    "RMSE": "${:,.2f}",
    "MAPE_Pct": "{:.2f}%"
}))

forecast_metrics_df.to_csv(REPORTS_DIR / "forecast_model_comparison.csv", index=False)

BEST_FORECAST_MODEL = forecast_metrics_df.iloc[0]["Model"]
print("Best test model by RMSE:", BEST_FORECAST_MODEL)

## 17. Actual vs predicted chart

In [ ]:
forecast_test_df = test[["Date", "Adj Close"]].copy()
forecast_test_df = forecast_test_df.rename(columns={"Adj Close": "ActualAdjClose"})
forecast_test_df["NaivePrediction"] = naive_predictions
forecast_test_df["ARIMAPrediction"] = arima_predictions

plt.figure(figsize=(12, 6))
plt.plot(forecast_test_df["Date"], forecast_test_df["ActualAdjClose"], label="Actual")
plt.plot(forecast_test_df["Date"], forecast_test_df["NaivePrediction"], label="Naive")
plt.plot(forecast_test_df["Date"], forecast_test_df["ARIMAPrediction"], label="ARIMA")
plt.title(f"{PRIMARY_TICKER}: Actual vs Forecast — Test Period")
plt.xlabel("Date")
plt.ylabel("Adjusted Close ($)")
plt.legend()
plt.tight_layout()
plt.savefig(IMAGES_DIR / "forecast_vs_actual.png", dpi=160, bbox_inches="tight")
plt.show()

## 18. Future 20-business-day forecast

We refit ARIMA to the full historical SPY series and generate a short-horizon model forecast.

Future dates use generic business days, so this is an approximation of trading dates rather than an official exchange calendar.

In [ ]:
FULL_ARIMA_ORDER = (5, 1, 0)
future_model = ARIMA(
    spy["Adj Close"].astype(float),
    order=FULL_ARIMA_ORDER
).fit()

FUTURE_DAYS = 20
future_values = future_model.forecast(steps=FUTURE_DAYS).to_numpy()

last_date = spy["Date"].max()
future_dates = pd.bdate_range(last_date + pd.Timedelta(days=1), periods=FUTURE_DAYS)

future_forecast_df = pd.DataFrame({
    "Date": future_dates,
    "Ticker": PRIMARY_TICKER,
    "ForecastAdjClose": future_values,
    "ForecastType": "ARIMA Model Forecast"
})

display(future_forecast_df)

joblib.dump(future_model, MODELS_DIR / "forecasting_model.pkl")

## 19. Save forecast outputs

In [ ]:
# Historical test predictions
forecast_test_long = pd.DataFrame({
    "Date": np.tile(forecast_test_df["Date"].to_numpy(), 2),
    "Ticker": PRIMARY_TICKER,
    "ActualAdjClose": np.tile(forecast_test_df["ActualAdjClose"].to_numpy(), 2),
    "Model": np.repeat(["Naive Previous Close", "ARIMA(5,1,0)"], len(forecast_test_df)),
    "PredictedAdjClose": np.concatenate([
        forecast_test_df["NaivePrediction"].to_numpy(),
        forecast_test_df["ARIMAPrediction"].to_numpy()
    ]),
    "RecordType": "Historical Test Forecast"
})

forecast_path = DATA_DIR / "forecast_predictions.csv"
forecast_test_long.to_csv(forecast_path, index=False)

future_path = DATA_DIR / "future_20day_forecast.csv"
future_forecast_df.to_csv(future_path, index=False)

print("Saved:", forecast_path)
print("Saved:", future_path)

## 20. Build Power BI-ready datasets

In [ ]:
market_powerbi = clean_df.copy()

# Helpful percentages converted to plain numeric decimals; Power BI will format them.
market_powerbi.to_csv(POWERBI_DIR / "market_powerbi.csv", index=False)

kpi_df.to_csv(POWERBI_DIR / "market_kpi_summary.csv", index=False)
yearly_df.to_csv(POWERBI_DIR / "yearly_performance.csv", index=False)
forecast_test_long.to_csv(POWERBI_DIR / "forecast_test_predictions.csv", index=False)
future_forecast_df.to_csv(POWERBI_DIR / "future_20day_forecast.csv", index=False)

print("Power BI files created.")

## 21. Generate MySQL-ready SQL scripts

In [ ]:
database_setup_sql = r"""
CREATE DATABASE IF NOT EXISTS stock_market_analytics;
USE stock_market_analytics;

CREATE TABLE IF NOT EXISTS market_data (
    trade_date DATE NOT NULL,
    ticker VARCHAR(10) NOT NULL,
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    adj_close DOUBLE,
    volume BIGINT,
    daily_return DOUBLE,
    log_return DOUBLE,
    cumulative_return DOUBLE,
    ma20 DOUBLE,
    ma50 DOUBLE,
    rolling_vol20 DOUBLE,
    running_peak DOUBLE,
    drawdown DOUBLE,
    volume_change DOUBLE,
    calendar_year INT,
    calendar_month VARCHAR(7),
    month_name VARCHAR(20),
    day_of_week VARCHAR(20),
    PRIMARY KEY (ticker, trade_date)
);

CREATE INDEX idx_market_date ON market_data(trade_date);
CREATE INDEX idx_market_ticker ON market_data(ticker);
"""

cleaning_queries_sql = r"""
-- Basic data-quality checks
SELECT ticker, trade_date, COUNT(*) AS row_count
FROM market_data
GROUP BY ticker, trade_date
HAVING COUNT(*) > 1;

SELECT *
FROM market_data
WHERE adj_close <= 0
   OR close_price <= 0
   OR volume < 0;

SELECT
    ticker,
    MIN(trade_date) AS first_date,
    MAX(trade_date) AS last_date,
    COUNT(*) AS trading_days
FROM market_data
GROUP BY ticker;
"""

analysis_queries_sql = r"""
-- 1. Latest historical price
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY ticker
            ORDER BY trade_date DESC
        ) AS rn
    FROM market_data
)
SELECT ticker, trade_date, adj_close
FROM ranked
WHERE rn = 1
ORDER BY ticker;

-- 2. Best and worst daily return
SELECT
    ticker,
    MAX(daily_return) AS best_daily_return,
    MIN(daily_return) AS worst_daily_return
FROM market_data
GROUP BY ticker;

-- 3. Average volume
SELECT
    ticker,
    AVG(volume) AS avg_daily_volume
FROM market_data
GROUP BY ticker
ORDER BY avg_daily_volume DESC;

-- 4. Positive vs negative days
SELECT
    ticker,
    SUM(CASE WHEN daily_return > 0 THEN 1 ELSE 0 END) AS positive_days,
    SUM(CASE WHEN daily_return < 0 THEN 1 ELSE 0 END) AS negative_days
FROM market_data
GROUP BY ticker;

-- 5. Days above 50-day moving average
SELECT
    ticker,
    SUM(CASE WHEN adj_close > ma50 THEN 1 ELSE 0 END) AS days_above_ma50,
    COUNT(ma50) AS valid_ma50_days,
    1.0 * SUM(CASE WHEN adj_close > ma50 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(ma50), 0) AS pct_above_ma50
FROM market_data
GROUP BY ticker
ORDER BY pct_above_ma50 DESC;

-- 6. Large move days
SELECT
    ticker,
    COUNT(*) AS return_days,
    SUM(CASE WHEN ABS(daily_return) > 0.03 THEN 1 ELSE 0 END) AS large_move_days,
    1.0 * SUM(CASE WHEN ABS(daily_return) > 0.03 THEN 1 ELSE 0 END)
        / COUNT(*) AS large_move_pct
FROM market_data
WHERE daily_return IS NOT NULL
GROUP BY ticker
ORDER BY large_move_pct DESC;

-- 7. Day-of-week average return
SELECT
    ticker,
    day_of_week,
    AVG(daily_return) AS average_daily_return
FROM market_data
WHERE daily_return IS NOT NULL
GROUP BY ticker, day_of_week
ORDER BY ticker, average_daily_return DESC;

-- 8. Top volume days
SELECT
    trade_date,
    ticker,
    volume,
    adj_close
FROM market_data
ORDER BY volume DESC
LIMIT 10;
"""

(SQL_DIR / "database_setup.sql").write_text(database_setup_sql, encoding="utf-8")
(SQL_DIR / "cleaning_queries.sql").write_text(cleaning_queries_sql, encoding="utf-8")
(SQL_DIR / "analysis_queries.sql").write_text(analysis_queries_sql, encoding="utf-8")

print("MySQL-ready scripts created.")

## 22. Business findings

In [ ]:
top_return = kpi_df.iloc[0]
most_volatile = kpi_df.sort_values("AnnualizedVolatility", ascending=False).iloc[0]
deepest_drawdown = kpi_df.sort_values("MaximumDrawdown").iloc[0]
best_risk_adjusted = kpi_df.sort_values("SharpeStyleRatio_0pctRF", ascending=False).iloc[0]

findings = f'''
STOCK MARKET ANALYTICS & FORECASTING — BUSINESS FINDINGS
========================================================

Historical period:
2020-01-01 to 2025-12-31

Securities:
{", ".join(TICKERS)}

Historical market analytics
---------------------------
Highest total return:
{top_return["Ticker"]} — {top_return["TotalReturn"]:.2%}

Highest annualised volatility:
{most_volatile["Ticker"]} — {most_volatile["AnnualizedVolatility"]:.2%}

Deepest maximum drawdown:
{deepest_drawdown["Ticker"]} — {deepest_drawdown["MaximumDrawdown"]:.2%}

Highest Sharpe-style ratio using 0% risk-free-rate assumption:
{best_risk_adjusted["Ticker"]} — {best_risk_adjusted["SharpeStyleRatio_0pctRF"]:.2f}

Forecasting
-----------
Primary ticker:
{PRIMARY_TICKER}

Test period:
Last {TEST_DAYS} historical trading observations.

Best forecasting model by test RMSE:
{BEST_FORECAST_MODEL}

Model comparison:
{forecast_metrics_df.to_string(index=False)}

Interpretation
--------------
1. Historical returns vary materially across securities.
2. Higher return is not automatically better because volatility and drawdown differ.
3. Cross-security correlation should be evaluated using returns rather than raw prices.
4. Rolling volatility changes over time and can rise sharply during stressed periods.
5. A naive forecast is a strong baseline for short-horizon asset-price forecasting.
6. Forecast accuracy should be judged out-of-sample using chronological data.
7. Forecasts are model outputs, not guarantees or investment recommendations.

Important disclaimer
--------------------
This project is educational analytics only and does not provide financial advice.
Historical performance and statistical forecasts do not guarantee future results.
'''

print(findings)
(REPORTS_DIR / "business_findings.txt").write_text(findings, encoding="utf-8")

## 23. Save reusable Python pipeline

In [ ]:
pipeline_code = r"""# Stock Market Analytics Pipeline
# Generated by Hassan Project 4 notebook.

import numpy as np
import pandas as pd

def engineer_market_features(df):
    df = df.sort_values(["Ticker", "Date"]).copy()

    def _features(g):
        g = g.sort_values("Date").copy()
        g["DailyReturn"] = g["Adj Close"].pct_change()
        g["LogReturn"] = np.log(g["Adj Close"] / g["Adj Close"].shift(1))
        g["CumulativeReturn"] = (1 + g["DailyReturn"].fillna(0)).cumprod() - 1
        g["MA20"] = g["Adj Close"].rolling(20).mean()
        g["MA50"] = g["Adj Close"].rolling(50).mean()
        g["RollingVol20"] = g["DailyReturn"].rolling(20).std() * np.sqrt(252)
        g["RunningPeak"] = g["Adj Close"].cummax()
        g["Drawdown"] = g["Adj Close"] / g["RunningPeak"] - 1
        return g

    return (
        df.groupby("Ticker", group_keys=False)
          .apply(_features)
          .reset_index(drop=True)
    )
"""

(SRC_DIR / "market_analytics_pipeline.py").write_text(
    pipeline_code,
    encoding="utf-8"
)

print("Saved reusable pipeline.")

## 24. Power BI dashboard plan

Use the CSV files in `project_4_outputs/powerbi/`.

### Page 1 — Market Overview
- Latest historical price
- Total return
- Average volume
- Cumulative-return trend
- Price trend
- Ticker / year slicers

### Page 2 — Returns & Risk
- Annualised return
- Annualised volatility
- Maximum drawdown
- Sharpe-style ratio
- Rolling volatility
- Risk-return scatter

### Page 3 — Security Comparison
- Ticker ranking
- Yearly returns
- Benchmark-relative analysis
- Correlation matrix
- Volume comparison

### Page 4 — Forecast & Model Evaluation
- SPY actual vs predicted
- Naive vs ARIMA metrics
- Future 20-business-day model forecast
- MAE / RMSE / MAPE
- Forecast methodology and disclaimer

## 25. Create requirements and ZIP all outputs

In [ ]:
requirements = '''pandas
numpy
matplotlib
yfinance
statsmodels
scikit-learn
joblib
'''

(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

generated_readme = f'''# Stock Market Analytics & Time Series Forecasting

Generated by `Hassan_Project_4_Stock_Market_Analytics_Forecasting_READY.ipynb`.

## Historical period
2020-01-01 to 2025-12-31

## Securities
{", ".join(TICKERS)}

## Forecast ticker
{PRIMARY_TICKER}

## Best test forecast by RMSE
{BEST_FORECAST_MODEL}

## Main outputs
- cleaned multi-ticker market dataset
- financial KPI summary
- yearly performance
- correlation matrix
- SQLite database
- MySQL-ready SQL scripts
- test forecast predictions
- future 20-business-day forecast
- saved ARIMA model
- Power BI-ready CSV files
- charts and business findings

## Disclaimer
Educational analytics only. Not financial advice.
'''

(PROJECT_DIR / "README_GENERATED.md").write_text(
    generated_readme,
    encoding="utf-8"
)

import shutil

zip_path = shutil.make_archive(
    "Hassan_Project_4_Colab_Outputs",
    "zip",
    PROJECT_DIR
)

print("Created ZIP:", zip_path)
print("\nProject 4 analysis is complete.")
print("Download the ZIP from the Colab Files panel.")

# Finished

If all cells ran successfully, the Python, SQL and forecasting stage of Project 4 is complete.

### Main generated files

- `project_4_outputs/data/market_data_cleaned.csv`
- `project_4_outputs/data/market_kpi_summary.csv`
- `project_4_outputs/data/correlation_matrix.csv`
- `project_4_outputs/data/yearly_performance.csv`
- `project_4_outputs/data/forecast_predictions.csv`
- `project_4_outputs/data/future_20day_forecast.csv`
- `project_4_outputs/market_analytics.sqlite`
- `project_4_outputs/models/forecasting_model.pkl`
- `project_4_outputs/sql/database_setup.sql`
- `project_4_outputs/sql/cleaning_queries.sql`
- `project_4_outputs/sql/analysis_queries.sql`
- `project_4_outputs/powerbi/market_powerbi.csv`
- `project_4_outputs/reports/business_findings.txt`
- charts in `project_4_outputs/images/`
- `Hassan_Project_4_Colab_Outputs.zip`

Next stage: **Power BI dashboard + GitHub packaging**.